In [13]:
import re
from pathlib import Path
import pandas as pd

# -----------------------------
# 1) File discovery
# -----------------------------
ROUND_FILE_RE = re.compile(
    r"""(?ix)
    ^(?:r|round|brainstormin\ round)?\s*(\d+)\.txt$
    """
)

def find_round_files(folder: str | Path) -> list[tuple[int, Path]]:
    folder = Path(folder)
    if not folder.exists():
        raise FileNotFoundError(f"Folder not found: {folder}")

    found = []
    for p in folder.iterdir():
        if not p.is_file():
            continue
        m = ROUND_FILE_RE.match(p.name)
        if m:
            found.append((int(m.group(1)), p))

    if not found:
        raise FileNotFoundError(
            f"No round .txt files found in {folder}. "
            "Expected names like r1.txt, r2.txt, round3.txt, etc."
        )

    return sorted(found, key=lambda x: x[0])


# -----------------------------
# 2) Parsing
# -----------------------------
LINE_RE = re.compile(
    r"""^
    \s*(?P<table>\d*)\s+
    (?P<player>.+?)\s{2,}
    (?P<opponent>.+?)\s{2,}
    (?P<points>\d+\s*-\s*\d+|\d+)\s*
    $""",
    re.VERBOSE,
)

def parse_round_file(path: Path, round_num: int) -> list[dict]:
    rows = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.rstrip("\n")

        # skip junk/header lines
        if not line.strip():
            continue
        if "Table" in line or set(line.strip()) <= {"-"}:
            continue
        if line.strip().lower().startswith("round"):
            continue

        m = LINE_RE.match(line)
        if not m:
            continue

        table = m.group("table").strip() or None
        player = m.group("player").strip()
        opponent = m.group("opponent").strip()
        points = m.group("points").replace(" ", "")

        rows.append(
            {
                "Round": round_num,
                "Table": table,
                "Player": player,
                "Opponent": opponent,
                "PointsRaw": points,
            }
        )
    return rows


def build_matches_df(folder: str | Path) -> pd.DataFrame:
    round_files = find_round_files(folder)

    records = []
    for rnd, path in round_files:
        records.extend(parse_round_file(path, rnd))

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError("Parsed 0 rows. Check file formatting or regex.")

    # numeric parsing:
    # - we want the *cumulative total* for the player after this round.
    # - in your files: "6-3", "9-6", etc. => take the left side (player's total)
    df["PlayerTotal"] = (
        df["PointsRaw"]
        .astype(str)
        .str.extract(r"^\s*(\d+)")
        .astype(int)
    )

    # normalize bye marker
    df["IsBye"] = df["Opponent"].str.strip().eq("***Bye***")

    return df.sort_values(["Round", "Table", "Player"]).reset_index(drop=True)


# -----------------------------
# 3) Derive outcomes by comparing to previous totals
# -----------------------------
def interpret_results(df_matches: pd.DataFrame) -> pd.DataFrame:
    # running totals per player
    prev_totals: dict[str, int] = {}

    out = []
    for rnd in sorted(df_matches["Round"].unique()):
        df_r = df_matches[df_matches["Round"] == rnd]

        for _, row in df_r.iterrows():
            player = row["Player"]
            opponent = row["Opponent"]
            new_total = int(row["PlayerTotal"])
            prev_total = int(prev_totals.get(player, 0))
            gain = new_total - prev_total

            # update running totals (even for byes, so next round math works)
            prev_totals[player] = new_total

            # ignore byes in outcome calculations (but keep row if you want)
            if row["IsBye"]:
                outcome = "Bye"
            else:
                if gain == 3:
                    outcome = "Win"
                elif gain == 1:
                    outcome = "Tie"
                elif gain == 0:
                    outcome = "Loss"
                else:
                    outcome = f"Check({gain})"

            out.append(
                {
                    "Round": rnd,
                    "Table": row["Table"],
                    "Player": player,
                    "Opponent": opponent,
                    "PrevTotal": prev_total,
                    "NewTotal": new_total,
                    "Gain": gain,
                    "Outcome": outcome,
                    "IsBye": bool(row["IsBye"]),
                }
            )

    return pd.DataFrame(out).sort_values(["Round", "Table", "Player"]).reset_index(drop=True)


# -----------------------------
# 4) Optional: turn player-rows into match-level rows (Player A vs Player B)
#    (useful for Elo updates)
# -----------------------------
def build_matchups(df_matches: pd.DataFrame) -> pd.DataFrame:
    """
    Creates one row per actual match (ignores byes), by pairing reciprocal rows
    within the same round (Player vs Opponent).
    """
    df = df_matches[~df_matches["IsBye"]].copy()

    # canonical key so A-vs-B and B-vs-A map together
    def key(a, b, rnd):
        lo, hi = sorted([a.strip(), b.strip()])
        return (rnd, lo, hi)

    df["MatchKey"] = [key(a, b, r) for a, b, r in zip(df["Player"], df["Opponent"], df["Round"])]

    grouped = df.groupby("MatchKey", as_index=False)

    match_rows = []
    for _, g in grouped:
        if len(g) != 2:
            # if something weird happens (missing reciprocal row), keep it visible
            match_rows.append(
                {
                    "Round": int(g["Round"].iloc[0]),
                    "PlayerA": g["Player"].iloc[0],
                    "PlayerB": g["Opponent"].iloc[0],
                    "A_Total": int(g["PlayerTotal"].iloc[0]),
                    "B_Total": None,
                    "Status": f"Unpaired({len(g)})",
                }
            )
            continue

        r0 = g.iloc[0]
        r1 = g.iloc[1]

        # r0 is "Player" vs "Opponent", and r1 should be reverse.
        # Determine winner by comparing *gain* from previous round is better,
        # but we can also just use inferred deltas from totals if you want later.
        match_rows.append(
            {
                "Round": int(r0["Round"]),
                "PlayerA": r0["Player"],
                "PlayerB": r0["Opponent"],
                "A_Total": int(r0["PlayerTotal"]),
                "B_Total": int(r1["PlayerTotal"]),
                "Status": "OK",
            }
        )

    return pd.DataFrame(match_rows).sort_values(["Round", "PlayerA", "PlayerB"]).reset_index(drop=True)


# -----------------------------
# Usage
# -----------------------------
folder = r"2_13_26"  # <-- set this

df_matches = build_matches_df(folder)
df_results = interpret_results(df_matches)
df_matchups = build_matchups(df_matches)

print(f"Parsed rows: {len(df_matches)}")
print(f"Result rows: {len(df_results)}")
print(f"Matchup rows: {len(df_matchups)}")

# If you want to ignore byes completely in df_results:
df_results_no_byes = df_results[~df_results["IsBye"]].copy()

# Save
df_matches.to_csv(Path(folder) / "matches_parsed.csv", index=False)
df_results.to_csv(Path(folder) / "results_interpreted.csv", index=False)
df_matchups.to_csv(Path(folder) / "matchups.csv", index=False)

Parsed rows: 91
Result rows: 91
Matchup rows: 88


In [16]:
import pandas as pd
from pathlib import Path

# =========================
# CONFIG
# =========================
RESULTS_CSV = r"2_13_26\results_interpreted.csv"  # <-- change to your file
INITIAL_ELO = 1500.0
K_FACTOR = 32.0

OUT_LEADERBOARD = "elo_leaderboard.csv"
OUT_MATCHUPS = "elo_matchups.csv"
OUT_MATCH_HISTORY = "elo_match_history.csv"
OUT_LADDER_BY_ROUND = "elo_ladder_by_round.csv"


# =========================
# Elo helpers
# =========================
def expected_score(r_a: float, r_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def elo_update(r_a: float, r_b: float, s_a: float, k: float) -> tuple[float, float]:
    e_a = expected_score(r_a, r_b)
    e_b = 1.0 - e_a
    s_b = 1.0 - s_a
    return (r_a + k * (s_a - e_a), r_b + k * (s_b - e_b))


# =========================
# Load + normalize interpreted results CSV
# =========================
def load_results_interpreted(csv_path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    # Normalize column names (common variants)
    rename_map = {}
    for c in df.columns:
        lc = c.strip().lower()
        if lc == "round":
            rename_map[c] = "Round"
        elif lc == "player":
            rename_map[c] = "Player"
        elif lc == "opponent":
            rename_map[c] = "Opponent"
        elif lc in ("gain", "delta", "points_gained"):
            rename_map[c] = "Gain"
        elif lc in ("outcome", "result"):
            rename_map[c] = "Outcome"
        elif lc in ("isbye", "is_bye", "bye"):
            rename_map[c] = "IsBye"

    df = df.rename(columns=rename_map)

    required = {"Round", "Player", "Opponent"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns in results CSV: {sorted(missing)}")

    # Ensure types/cleanup
    df["Round"] = df["Round"].astype(int)
    df["Player"] = df["Player"].astype(str).str.strip()
    df["Opponent"] = df["Opponent"].astype(str).str.strip()

    # IsBye handling
    if "IsBye" not in df.columns:
        df["IsBye"] = df["Opponent"].eq("***Bye***")
    else:
        # coerce to bool
        df["IsBye"] = df["IsBye"].astype(bool)

    # Outcome mapping (if not present, infer from Gain when possible)
    if "Outcome" not in df.columns:
        if "Gain" not in df.columns:
            raise KeyError("Need either Outcome column or Gain column in results CSV.")
        df["Outcome"] = df["Gain"].map({3: "Win", 1: "Tie", 0: "Loss"}).fillna("Check")

    # Gain is optional, but useful for diagnostics
    if "Gain" not in df.columns:
        # Create Gain from Outcome if absent
        df["Gain"] = df["Outcome"].map({"Win": 3, "Tie": 1, "Loss": 0}).fillna(pd.NA)

    return df


# =========================
# Build matchups (one row per actual match)
# =========================
def build_matchups_from_results(df_results: pd.DataFrame) -> pd.DataFrame:
    """
    Uses the interpreted results (Player/Opponent/Outcome) to create one row per match.
    Skips byes.
    """
    df = df_results[~df_results["IsBye"]].copy()

    # Canonical key so A-vs-B and B-vs-A pair up
    def mk_key(rnd, a, b):
        lo, hi = sorted([a, b])
        return (int(rnd), lo, hi)

    df["MatchKey"] = [mk_key(r, a, b) for r, a, b in zip(df["Round"], df["Player"], df["Opponent"])]

    rows = []
    for (rnd, lo, hi), g in df.groupby("MatchKey"):
        # Expect exactly 2 rows (A row + B row). If not, keep a diagnostic row.
        if len(g) != 2:
            rows.append({
                "Round": rnd,
                "PlayerA": lo,
                "PlayerB": hi,
                "ScoreA": None,
                "Status": f"Unpaired({len(g)})",
            })
            continue

        # Identify which row corresponds to lo playing hi
        row_lo = g[(g["Player"] == lo) & (g["Opponent"] == hi)]
        row_hi = g[(g["Player"] == hi) & (g["Opponent"] == lo)]

        if len(row_lo) != 1 or len(row_hi) != 1:
            rows.append({
                "Round": rnd,
                "PlayerA": lo,
                "PlayerB": hi,
                "ScoreA": None,
                "Status": "BadPairingRows",
            })
            continue

        o_lo = row_lo["Outcome"].iloc[0]
        o_hi = row_hi["Outcome"].iloc[0]

        # Convert outcome for PlayerA=lo into Elo score
        # Win=1, Tie=0.5, Loss=0
        map_score = {"Win": 1.0, "Tie": 0.5, "Loss": 0.0}
        score_a = map_score.get(str(o_lo), None)

        # Sanity check: if lo says Win, hi should say Loss, etc.
        # If score_a is None or mismatch, flag it.
        ok = True
        if score_a is None:
            ok = False
        else:
            expected_other = {"Win": "Loss", "Loss": "Win", "Tie": "Tie"}.get(str(o_lo))
            if expected_other is None or str(o_hi) != expected_other:
                ok = False

        rows.append({
            "Round": rnd,
            "PlayerA": lo,
            "PlayerB": hi,
            "ScoreA": score_a,
            "OutcomeA": o_lo,
            "OutcomeB": o_hi,
            "Status": "OK" if ok else "CheckOutcomeMismatch",
        })

    df_matchups = pd.DataFrame(rows).sort_values(["Round", "PlayerA", "PlayerB"]).reset_index(drop=True)
    return df_matchups


# =========================
# Elo with ladder snapshot for EVERY player EACH round
# =========================
def run_elo_ladder(
    df_matchups: pd.DataFrame,
    initial_elo: float = 1500.0,
    k: float = 32.0,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      leaderboard (final round standings),
      match_history (one row per match, chronological),
      ladder_by_round (one row per player per round, even if they didn't play)
    """

    # Keep only usable matches
    df_ok = df_matchups[df_matchups["Status"] == "OK"].copy()
    if df_ok.empty:
        # Show what went wrong
        raise ValueError(
            "No OK matchups found. Inspect df_matchups['Status'] counts:\n"
            f"{df_matchups['Status'].value_counts(dropna=False)}"
        )

    players = sorted(set(df_ok["PlayerA"]) | set(df_ok["PlayerB"]))
    rounds = sorted(df_ok["Round"].unique())

    ratings = {p: float(initial_elo) for p in players}
    stats = {p: {"Games": 0, "Wins": 0, "Losses": 0, "Ties": 0} for p in players}

    match_history = []
    ladder_rows = []
    match_index = 0

    for rnd in rounds:
        df_r = df_ok[df_ok["Round"] == rnd].copy()

        # Run matches in deterministic order
        df_r = df_r.sort_values(["PlayerA", "PlayerB"]).reset_index(drop=True)

        for _, m in df_r.iterrows():
            a = m["PlayerA"]
            b = m["PlayerB"]
            s_a = float(m["ScoreA"])

            pre_a, pre_b = ratings[a], ratings[b]
            post_a, post_b = elo_update(pre_a, pre_b, s_a, k)

            ratings[a], ratings[b] = post_a, post_b

            # update stats
            stats[a]["Games"] += 1
            stats[b]["Games"] += 1
            if s_a == 1.0:
                stats[a]["Wins"] += 1
                stats[b]["Losses"] += 1
                outcome = f"{a} beat {b}"
            elif s_a == 0.0:
                stats[a]["Losses"] += 1
                stats[b]["Wins"] += 1
                outcome = f"{b} beat {a}"
            else:
                stats[a]["Ties"] += 1
                stats[b]["Ties"] += 1
                outcome = f"{a} tied {b}"

            match_index += 1
            match_history.append({
                "MatchIndex": match_index,
                "Round": rnd,
                "PlayerA": a,
                "PlayerB": b,
                "ScoreA": s_a,
                "PreA": pre_a,
                "PreB": pre_b,
                "PostA": post_a,
                "PostB": post_b,
                "K": k,
                "OutcomeText": outcome,
            })

        # Snapshot ladder for ALL players for this round
        for p in players:
            ladder_rows.append({
                "Round": rnd,
                "Player": p,
                "Elo": ratings[p],
                "Games": stats[p]["Games"],
                "Wins": stats[p]["Wins"],
                "Losses": stats[p]["Losses"],
                "Ties": stats[p]["Ties"],
            })

    match_history_df = pd.DataFrame(match_history)
    ladder_by_round = pd.DataFrame(ladder_rows)

    # Add DeltaElo (change since last round) for each player
    ladder_by_round = ladder_by_round.sort_values(["Player", "Round"]).reset_index(drop=True)
    ladder_by_round["DeltaElo"] = ladder_by_round.groupby("Player")["Elo"].diff().fillna(0.0)

    # Pretty rounding
    ladder_by_round["Elo"] = ladder_by_round["Elo"].round(2)
    ladder_by_round["DeltaElo"] = ladder_by_round["DeltaElo"].round(2)

    # Final leaderboard is last round snapshot
    last_round = max(rounds)
    leaderboard = (
        ladder_by_round[ladder_by_round["Round"] == last_round]
        .sort_values(["Elo", "Wins"], ascending=[False, False])
        .reset_index(drop=True)
    )

    return leaderboard, match_history_df, ladder_by_round


def print_leaders(leaderboard: pd.DataFrame, top_n: int = 10):
    cols = ["Player", "Elo", "Games", "Wins", "Losses", "Ties"]
    show = leaderboard[cols].head(top_n).copy()
    print(show.to_string(index=False))


# =========================
# MAIN
# =========================
def main():
    df_results = load_results_interpreted(RESULTS_CSV)
    print("Loaded results_interpreted.csv:", df_results.shape)
    print("Outcome counts:\n", df_results["Outcome"].value_counts(dropna=False))
    print("Bye rows:", int(df_results["IsBye"].sum()))

    df_matchups = build_matchups_from_results(df_results)
    print("\nMatchup status counts:\n", df_matchups["Status"].value_counts(dropna=False))

    # Save matchups so you can inspect them
    df_matchups.to_csv(OUT_MATCHUPS, index=False)

    leaderboard, match_history, ladder_by_round = run_elo_ladder(
        df_matchups,
        initial_elo=INITIAL_ELO,
        k=K_FACTOR
    )

    # Save outputs
    leaderboard.to_csv(OUT_LEADERBOARD, index=False)
    match_history.to_csv(OUT_MATCH_HISTORY, index=False)
    ladder_by_round.to_csv(OUT_LADDER_BY_ROUND, index=False)

    print("\n=== LEADERS ===")
    print_leaders(leaderboard, top_n=15)

    print("\nSaved:")
    print(" -", OUT_MATCHUPS)
    print(" -", OUT_LEADERBOARD)
    print(" -", OUT_MATCH_HISTORY)
    print(" -", OUT_LADDER_BY_ROUND)


if __name__ == "__main__":
    main()

Loaded results_interpreted.csv: (91, 9)
Outcome counts:
 Outcome
Win         43
Loss        29
Check(6)     8
Tie          3
Bye          3
Check(4)     3
Check(7)     1
Check(9)     1
Name: count, dtype: int64
Bye rows: 3

Matchup status counts:
 Status
Unpaired(1)    88
Name: count, dtype: int64


ValueError: No OK matchups found. Inspect df_matchups['Status'] counts:
Status
Unpaired(1)    88
Name: count, dtype: int64

KeyError: 'Rating'

In [12]:
df_results_no_byes = df_results[~df_results["IsBye"]].copy()

df_matchups = build_matchups_with_scores(df_results_no_byes)
display(df_matchups)

final_ratings, df_elo_history = run_elo(
    df_matchups,
    initial_rating=1500,
    base_k=32,
    use_adaptive_k=True,   # or False for constant K
)

display(final_ratings.head(20))
display(df_elo_history.head(20))

,Round,PlayerA,PlayerB,GainA,GainB,ScoreA,Winner,Status
0,1,Autumn Garofalo,Daniel Hernandez,3.0,None,None,None,Unpaired(1)
1,1,Becca Blackwood,Karl Rookey,0.0,None,None,None,Unpaired(1)
2,1,Dani Masom,Stella Burfeind,0.0,None,None,None,Unpaired(1)
3,1,Daniel Cotto,Reya Truher,0.0,None,None,None,Unpaired(1)
4,1,David Van Hise,Hex Dowd,3.0,None,None,None,Unpaired(1)
...,...,...,...,...,...,...,...,...
83,5,Pan Rasking,Mike Adams,4.0,None,None,None,Unpaired(1)
84,5,Philip Petronis,John Larochelle,1.0,None,None,None,Unpaired(1)
85,5,Reya Truher,Isharah Considine,0.0,None,None,None,Unpaired(1)
86,5,Stella Burfeind,Amy Altman,0.0,None,None,None,Unpaired(1)


,Player,Rating,Games


""


In [11]:
import pandas as pd

def expected_score(r_a: float, r_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def elo_update(r_a: float, r_b: float, s_a: float, k: float) -> tuple[float, float]:
    e_a = expected_score(r_a, r_b)
    s_b = 1.0 - s_a
    e_b = 1.0 - e_a
    return (r_a + k * (s_a - e_a), r_b + k * (s_b - e_b))

def run_running_elo(
    df_matchups: pd.DataFrame,
    initial_rating: float = 1500.0,
    k: float = 32.0,
    sort_cols: list[str] = None,   # e.g. ["Round", "Table"] if you have Table too
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - leaderboard_df: Player, Rating, Games, Wins, Losses, Ties
      - history_df: one row per match with pre/post ratings + match index
    """

    if sort_cols is None:
        sort_cols = ["Round"]

    df = df_matchups.copy()
    if "Status" in df.columns:
        df = df[df["Status"] == "OK"].copy()

    # deterministic processing order
    df = df.sort_values(sort_cols).reset_index(drop=True)

    ratings: dict[str, float] = {}
    stats: dict[str, dict] = {}  # games/w/l/t

    history_rows = []
    match_index = 0  # <-- global running count across ALL rounds

    def ensure_player(p: str):
        if p not in ratings:
            ratings[p] = initial_rating
        if p not in stats:
            stats[p] = {"Games": 0, "Wins": 0, "Losses": 0, "Ties": 0}

    for _, m in df.iterrows():
        a = str(m["PlayerA"]).strip()
        b = str(m["PlayerB"]).strip()
        s_a = float(m["ScoreA"])
        rnd = int(m["Round"]) if "Round" in m else None

        ensure_player(a)
        ensure_player(b)

        pre_a = ratings[a]
        pre_b = ratings[b]

        post_a, post_b = elo_update(pre_a, pre_b, s_a, k)

        # update ratings
        ratings[a] = post_a
        ratings[b] = post_b

        # update running stats
        stats[a]["Games"] += 1
        stats[b]["Games"] += 1

        if s_a == 1.0:
            stats[a]["Wins"] += 1
            stats[b]["Losses"] += 1
            outcome = "A_win"
        elif s_a == 0.0:
            stats[a]["Losses"] += 1
            stats[b]["Wins"] += 1
            outcome = "B_win"
        else:
            stats[a]["Ties"] += 1
            stats[b]["Ties"] += 1
            outcome = "Tie"

        match_index += 1

        history_rows.append({
            "MatchIndex": match_index,   # running count across all rounds
            "Round": rnd,
            "PlayerA": a,
            "PlayerB": b,
            "ScoreA": s_a,
            "Outcome": outcome,
            "PreA": pre_a,
            "PreB": pre_b,
            "PostA": post_a,
            "PostB": post_b,
            "K": k,
        })

    history_df = pd.DataFrame(history_rows)

    leaderboard_df = (
        pd.DataFrame([
            {
                "Player": p,
                "Rating": ratings[p],
                **stats[p],
            }
            for p in ratings.keys()
        ])
        .sort_values(["Rating", "Wins"], ascending=[False, False])
        .reset_index(drop=True)
    )

    return leaderboard_df, history_df


def print_leaders(leaderboard_df: pd.DataFrame, top_n: int = 10):
    cols = ["Player", "Rating", "Games", "Wins", "Losses", "Ties"]
    show = leaderboard_df[cols].head(top_n).copy()
    show["Rating"] = show["Rating"].round(1)
    print(show.to_string(index=False))


# ---------------------------
# USAGE
# ---------------------------
# df_matchups must already exist
leaderboard, elo_history = run_running_elo(
    df_matchups,
    initial_rating=1500,
    k=32,
    sort_cols=["Round"],  # add "Table" if you want within-round table order too
)

print_leaders(leaderboard, top_n=15)

# Save outputs
leaderboard.to_csv("elo_leaderboard.csv", index=False)
elo_history.to_csv("elo_history.csv", index=False)

KeyError: 'Rating'